In [1]:
import numpy as np
import math
import torch

# ——— Load and prepare q0, k0 as NumPy arrays ———
q_full = torch.load("subset_qk/block_1_q_proj_batch_6.pt", map_location="cpu")
k_full = torch.load("subset_qk/block_1_k_proj_batch_6.pt", map_location="cpu")

q = q_full[0]  # shape [L, d_model]
k = k_full[0]

L, d_model = q.shape
num_heads  = 32
d_head     = d_model // num_heads
POS = 0
# pick head 15 and first `sample` positions
sample = 4096
def sampling(q, k, sample, normalize=False):
    q0 = (
        q
        .view(L, num_heads, d_head)
        .permute(1, 0, 2)[POS, :sample]
        .numpy()
    )   # shape [sample, d_head]
    k0 = (
        k
        .view(L, num_heads, d_head)
        .permute(1, 0, 2)[POS, :sample]
        .numpy()
    )

    q0 = q0 / 128 ** 0.25
    k0 = k0 / 128 ** 0.25
    if normalize:
        
        q0 = unit_norm_normalize(q0)
        k0 = unit_norm_normalize(k0)
    return q0, k0

def true_softmax(q0, k0):
    dot = q0 @ k0.T
    true_val = np.exp(dot - dot.max(axis=1, keepdims=True))
    true_val /= true_val.sum(axis=1, keepdims=True)
    return true_val

def unit_norm_normalize(matrix):
    """
    对矩阵进行单位范数归一化（L2归一化）
    matrix: (N, M) 输入矩阵
    returns: (N, M) 归一化后的矩阵，每行的L2范数为1
    """
    # 计算每行的L2范数
    row_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    
    # 归一化
    normalized_matrix = matrix / row_norms
    
    return normalized_matrix


In [2]:
def report_error(record_approx_values, true_val):
    return torch.norm(torch.tensor(record_approx_values - true_val)) / torch.norm(torch.tensor(true_val))

def rfa_q_feature_mapping_blocks(q0, P=8, D=2000):
    """
    保持输入为P个块，每个块D个向量，输出展平为P*D维
    q0: (N, d) 查询数组
    P: 块数 (默认为8)
    D: 每个块的向量数 (默认为2000)
    shrink: 缩放因子
    returns: (N, P*D) 展平后的特征
    """
    # 1) 预处理和缩放
    X = (q0).astype(np.float64)   # (N, d)
    N, d = X.shape

    # 2) 为每个块生成随机权重
    w_blocks = np.sign(np.random.randn(P, D, d)).astype(np.float64)  # (P, D, d)

    # 3) 计算每个块的投影
    proj_blocks = np.zeros((N, P, D))  # (N, P, D)
    for p in range(P):
        proj_blocks[:, p, :] = X.dot(w_blocks[p].T)  # (N, D)

    # 4) 计算每个块的归一化因子
    facts = np.array([math.sqrt(math.factorial(p+1)) for p in range(P)],
                     dtype=np.float64)     # (P,)
    normalizer = np.sqrt(D, dtype=np.float64) * facts
    normalizer = normalizer.reshape(1, P, 1)  # (1, P, 1)

    # 5) 应用归一化
    phi_blocks = proj_blocks / normalizer  # (N, P, D)

    # 6) 展平为(N, P*D)
    phi_flat = phi_blocks.reshape(N, P*D)  # (N, 16000)
    return phi_flat

# Usage

In [49]:
P, D, d = 2, 800, 128
sample=4096
q0, k0 = sampling(q, k, sample)
v0 =(q0+ 4 * k0) /3

In [24]:
# get real result
real_result = true_softmax(q0, k0)
real_result = real_result @ v0

In [50]:
approx_result = np.zeros(q0.shape)

In [51]:
# get first row of result
phi_q0 = rfa_q_feature_mapping_blocks(q0, P, D) # Shape 4096, P* D
# result is 
phi_k0 = rfa_q_feature_mapping_blocks(k0, P, D) # Shape 4096, P* D
# Sha
phi_q0 += 1/math.sqrt(P*D) #麦克劳林展开的1，分摊在每个dim里
phi_k0 += 1/math.sqrt(P*D)
rest = np.zeros((phi_k0.shape[1], 128)) # Shape P* D, 128
for j in range(128):
    phi_k_v_j = np.zeros((1, phi_k0.shape[1])) # Shape 1, P* D 
    for i in range(4096):
        k_is = phi_k0[i]
        phi_k_v_j += k_is * v0[i, j] # V0 ij is scalar
    rest[:, j] = phi_k_v_j # Placement
# Get top of RF attention
top = phi_q0 @ rest # Shape 4096, P* D product  Shape P* D, 128 get 4096, 128

# Get Bottom of softmax kernel version random feature
row_sum = phi_k0.sum(axis=0)  # Shape  P* D, 1
for i in range(4096):
    bottom_i = phi_q0[i].dot(row_sum)
    approx_result[i] = top[i] / bottom_i #



In [52]:
report_error(approx_result, real_result)

tensor(0.1172, dtype=torch.float64)